# Compression-BPC baseline against language-model cross-entropy: Academic papers

This notebook preserves the existing research-paper PDF reading-order reconstruction, duplicate removal, subsection extraction, and complete-paragraph chunking pipeline. It scores identical chunks with the four language models used in the cross-model study and compression baselines.

In [6]:
# Install dependencies used by the PDF parser and GPT-2 scorer.
%pip install -q pymupdf transformers accelerate scipy zstandard requests

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import os
import re
import math
import json
import shutil
import hashlib
import traceback
import zipfile
import collections

import fitz
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display

DATASET_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/paper_compression_baseline")
EXTRACT_ROOT = OUTPUT_ROOT / "extracted_papers"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

MIN_WORDS = 200
MODEL_NAMES = {
    "gpt2": "gpt2",
    "pythia_410m": "EleutherAI/pythia-410m",
    "qwen2_5_0_5b": "Qwen/Qwen2.5-0.5B",
    "tinyllama_1_1b": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STRIDE = 512
LOWERCASE_FOR_LM = False
HISTOGRAM_METHOD = "fd"
MIN_BINS = 8
MAX_BINS = 30

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20261012
PRE_REGISTERED_MEAN_RHO_THRESHOLD = 0.85

HEADER_FRACTION = 0.055
FOOTER_FRACTION = 0.045
END_SECTION_PATTERNS = [
    r"^references\b", r"^bibliography\b", r"^acknowledg(?:e)?ments?\b",
    r"^credit authorship contribution statement\b", r"^declaration of competing interest\b",
    r"^declaration of interests\b", r"^data availability\b", r"^funding\b",
    r"^funding statement\b", r"^supplementary (?:data|material)\b", r"^appendix\b",
    r"^appendix [A-Z]\b", r"^author contributions?\b", r"^conflict of interest\b",
]
COMMON_UNNUMBERED_HEADINGS = {
    "abstract", "introduction", "background", "theoretical background", "literature review",
    "method", "methods", "methodology", "materials and methods", "results", "findings",
    "discussion", "general discussion", "discussion and conclusion", "conclusion", "conclusions",
    "limitations", "strengths & limitations", "current study", "the present study", "research methodology",
    "theoretical framework", "conceptual background", "recommendations", "implications", "future research",
}

print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)
print("Language models:", MODEL_NAMES)
print("Device:", DEVICE)
print("Minimum chunk size:", MIN_WORDS, "words")

Dataset: /kaggle/input
Output: /kaggle/working/paper_compression_baseline
Language models: {'gpt2': 'gpt2', 'pythia_410m': 'EleutherAI/pythia-410m', 'qwen2_5_0_5b': 'Qwen/Qwen2.5-0.5B', 'tinyllama_1_1b': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}
Device: cuda
Minimum chunk size: 200 words


## 1. Discover, deduplicate, and record the research-paper PDFs

In [8]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            block = stream.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def get_pdf_metadata(path):
    with fitz.open(str(path)) as doc:
        metadata = dict(doc.metadata or {})
        page_count = len(doc)
    return metadata, page_count

zip_files = sorted(DATASET_ROOT.rglob("*.zip"))
preferred_zips = [p for p in zip_files if "random_papers" in p.name.lower()]
if preferred_zips:
    zips_to_extract = preferred_zips
elif len(zip_files) == 1:
    zips_to_extract = zip_files
elif not zip_files:
    zips_to_extract = []
else:
    raise RuntimeError(
        "Multiple ZIP files were found under /kaggle/input and none is named random_papers. "
        "Keep only the research-paper ZIP in the Kaggle dataset, or name it random_papers.zip."
    )

for zip_path in zips_to_extract:
    extract_dir = EXTRACT_ROOT / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(extract_dir)
        marker.write_text("ok", encoding="utf-8")

pdf_candidates = set(DATASET_ROOT.rglob("*.pdf")) | set(EXTRACT_ROOT.rglob("*.pdf"))
all_pdf_paths = sorted((p.resolve() for p in pdf_candidates), key=lambda p: (p.name.lower(), str(p)))
if not all_pdf_paths:
    raise FileNotFoundError("No PDFs or ZIPs containing PDFs were found under /kaggle/input.")

manifest_rows = []
for path in all_pdf_paths:
    try:
        metadata, page_count = get_pdf_metadata(path)
        manifest_rows.append({
            "file": path.name,
            "path": str(path),
            "sha256": sha256_file(path),
            "pages": page_count,
            "title": metadata.get("title", "") or "",
            "author": metadata.get("author", "") or "",
            "subject": metadata.get("subject", "") or "",
            "read_status": "SUCCESS",
            "error": "",
        })
    except Exception as exc:
        manifest_rows.append({
            "file": path.name, "path": str(path), "sha256": "", "pages": np.nan,
            "title": "", "author": "", "subject": "", "read_status": "ERROR",
            "error": f"{type(exc).__name__}: {exc}",
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUTPUT_ROOT / "paper_manifest_all_files.csv", index=False, encoding="utf-8-sig")

usable = manifest_df[manifest_df["read_status"].eq("SUCCESS") & manifest_df["sha256"].ne("")].copy()
duplicate_report = (
    usable.groupby("sha256")["file"].agg(list).reset_index(name="files")
)
duplicate_report["n_files"] = duplicate_report["files"].str.len()
duplicate_report = duplicate_report[duplicate_report["n_files"] > 1].copy()
duplicate_report.to_csv(OUTPUT_ROOT / "exact_duplicate_report.csv", index=False, encoding="utf-8-sig")

# One unique PDF = one source-level random variable; exact duplicate files are excluded.
used = usable.sort_values(["file", "path"]).drop_duplicates("sha256", keep="first").copy()
used = used.sort_values(["file", "path"]).reset_index(drop=True)
used["paper_index"] = np.arange(1, len(used) + 1)
used["paper_id"] = [
    f"paper_{i:02d}_" + re.sub(r"[^A-Za-z0-9]+", "_", Path(name).stem).strip("_")
    for i, name in zip(used["paper_index"], used["file"])
]
paper_manifest = used[[
    "paper_index", "paper_id", "file", "path", "sha256", "pages", "title", "author", "subject"
]].copy()
paper_manifest.to_csv(OUTPUT_ROOT / "paper_manifest_used.csv", index=False, encoding="utf-8-sig")

print("PDF paths discovered:", len(all_pdf_paths))
print("Unique readable paper PDFs:", len(paper_manifest))
if not duplicate_report.empty:
    print("Exact duplicates excluded:")
    display(duplicate_report)
display(paper_manifest[["paper_index", "file", "pages", "title", "author"]])

PDF paths discovered: 29
Unique readable paper PDFs: 29


,paper_index,file,pages,title,author
0,1,1-s2.0-S0022440525001013-main.pdf,24,Reporting guidelines for single-case experimen...,Thomas R. Kratochwill
1,2,1-s2.0-S0022440525001189-main.pdf,20,Evaluating a pragmatic universal social behavi...,Wendy M. Reinke
2,3,1-s2.0-S0140197121001330-main.pdf,10,Relationships between neighborhood collective ...,Ji Hyeon Lee
3,4,1-s2.0-S0140197121001366-main.pdf,12,Measuring advanced theory of mind: Do story-ba...,Marta Białecka-Pikul
4,5,1-s2.0-S0160738326001635-main.pdf,12,Academic travel: A subcultural theory approach,Paolo Mura
5,6,1-s2.0-S0160738326001684-main.pdf,6,Valid uses of stated intentions in tourism res...,Sara Dolnicar
6,7,1-s2.0-S0190740926000447-main.pdf,15,Healthcare access among youth with disabilitie...,Leah P. Cheatham
7,8,1-s2.0-S019074092600071X-main.pdf,13,Children’s experience of a new approach to fam...,Srijani Datta
8,9,1-s2.0-S0195666326003582-main.pdf,12,"Capability, opportunity, and motivation for he...",Sebastian Isbanner
9,10,1-s2.0-S0195666326003594-main.pdf,11,Can liking be instructed? Evaluations of famil...,Katrijn Houben


## 2. Extract paper sections and paragraphs

In [9]:
WORD_RE = re.compile(
    r"(?u)\b[\wÀ-ÖØ-öø-ÿ]+(?:['’\-][\wÀ-ÖØ-öø-ÿ]+)*\b"
)

NUMBERED_HEADING_RE = re.compile(
    r"^(?:\d+(?:\.\d+)*|[IVXLC]+)\.?\s+\S+"
)

CAPTION_RE = re.compile(
    r"^(?:fig(?:ure)?|table)\s*[\dIVX]+[\.:]?",
    re.I,
)

PAGE_NUMBER_RE = re.compile(
    r"^\d{1,4}$"
)

END_HEADING_RX = [
    re.compile(
        pattern,
        re.I,
    )
    for pattern in END_SECTION_PATTERNS
]

def normalize_text(text):
    text = text.replace(
        "\xad",
        "",
    )
    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

def count_words(text):
    return len(
        WORD_RE.findall(text)
    )

def block_to_info(block):
    if block.get("type") != 0:
        return None

    spans = [
        sp
        for line in block.get(
            "lines",
            [],
        )
        for sp in line.get(
            "spans",
            [],
        )
        if sp.get("text", "").strip()
    ]

    if not spans:
        return None

    text = normalize_text(
        "".join(
            sp["text"]
            for sp in spans
        )
    )

    if not text:
        return None

    return {
        "text": text,
        "x0": min(
            sp["bbox"][0]
            for sp in spans
        ),
        "y0": min(
            sp["bbox"][1]
            for sp in spans
        ),
        "x1": max(
            sp["bbox"][2]
            for sp in spans
        ),
        "y1": max(
            sp["bbox"][3]
            for sp in spans
        ),
        "font": max(
            float(
                sp.get(
                    "size",
                    0.0,
                )
            )
            for sp in spans
        ),
        "bold": any(
            (
                "bold"
                in sp.get(
                    "font",
                    "",
                ).lower()
                or
                "semibold"
                in sp.get(
                    "font",
                    "",
                ).lower()
            )
            for sp in spans
        ),
    }

def page_blocks_reading_order(page):
    infos = []

    for block in page.get_text(
        "dict"
    ).get(
        "blocks",
        [],
    ):
        info = block_to_info(
            block
        )

        if info is not None:
            infos.append(info)

    if not infos:
        return []

    page_width = page.rect.width
    page_height = page.rect.height

    infos = [
        b
        for b in infos
        if (
            b["y0"]
            >= HEADER_FRACTION * page_height
            and
            b["y1"]
            <= (
                1.0 - FOOTER_FRACTION
            ) * page_height
        )
    ]

    if not infos:
        return []

    left = []
    right = []
    full = []

    for b in infos:
        if (
            b["x0"]
            < 0.08 * page_width
            and
            b["x1"]
            > 0.92 * page_width
        ):
            full.append(b)

        elif (
            b["x0"] + b["x1"]
        ) / 2 < page_width / 2:
            left.append(b)

        else:
            right.append(b)

    if left and right:
        ordered_columns = (
            sorted(
                left,
                key=lambda x: (
                    x["y0"],
                    x["x0"],
                ),
            )
            +
            sorted(
                right,
                key=lambda x: (
                    x["y0"],
                    x["x0"],
                ),
            )
        )

        if full:
            first_column_y = min(
                b["y0"]
                for b in ordered_columns
            )

            if (
                min(
                    b["y0"]
                    for b in full
                )
                <= first_column_y
            ):
                return (
                    sorted(
                        full,
                        key=lambda x: x["y0"],
                    )
                    + ordered_columns
                )

            return sorted(
                ordered_columns + full,
                key=lambda x: x["y0"],
            )

        return ordered_columns

    return sorted(
        infos,
        key=lambda x: (
            x["y0"],
            x["x0"],
        ),
    )

def is_end_heading(text):
    return any(
        rx.match(text)
        for rx in END_HEADING_RX
    )

def is_toc_line(text):
    return bool(
        re.search(
            r"\.{4,}\s*\d+\s*$",
            text,
        )
    )

def is_caption(text):
    return bool(
        CAPTION_RE.match(text)
    )

def is_standalone_equation(text):
    if len(text) < 8:
        return False

    words = count_words(text)

    letters = sum(
        ch.isalpha()
        for ch in text
    )

    symbols = sum(
        not (
            ch.isalnum()
            or ch.isspace()
            or ch in ".,;:'’()-[]"
        )
        for ch in text
    )

    return (
        words < 12
        and
        letters / max(
            len(text),
            1,
        ) < 0.35
        and
        symbols / max(
            len(text),
            1,
        ) > 0.12
    )

def is_heading(block):
    text = block["text"]
    lower = text.lower().rstrip(":")

    if len(text) > 180:
        return False

    if is_caption(text):
        return False

    if NUMBERED_HEADING_RE.match(
        text
    ):
        return True

    if lower in COMMON_UNNUMBERED_HEADINGS:
        return True

    if (
        block["bold"]
        and
        block["font"] >= 7.5
        and
        not text.endswith(
            (
                ".",
                ",",
                ";",
                "?",
                "!",
            )
        )
    ):
        return True

    return False

def extract_paper_sections(pdf_path):
    doc = fitz.open(
        str(pdf_path)
    )

    page_blocks = []
    short_text_frequency = (
        collections.Counter()
    )
    contents_pages = set()

    for page_number, page in enumerate(
        doc,
        start=1,
    ):
        blocks = (
            page_blocks_reading_order(
                page
            )
        )

        for block in blocks:
            block["page"] = page_number

            if len(block["text"]) <= 120:
                short_text_frequency[
                    block["text"].lower()
                ] += 1

            if re.fullmatch(
                r"contents",
                block["text"],
                flags=re.I,
            ):
                contents_pages.add(
                    page_number
                )

        page_blocks.append(
            blocks
        )

    doc.close()

    repeated_short_text = {
        text
        for text, count
        in short_text_frequency.items()
        if count >= max(
            2,
            int(
                0.35
                * len(page_blocks)
            ),
        )
    }

    skip_until_page = (
        max(contents_pages) + 1
        if contents_pages
        else 0
    )

    sections = []
    current = None
    started = False

    for blocks in page_blocks:
        if not blocks:
            continue

        page_number = blocks[0][
            "page"
        ]

        if page_number < skip_until_page:
            continue

        for block in blocks:
            text = normalize_text(
                block["text"]
            )

            lower = text.lower()

            if not text:
                continue

            if PAGE_NUMBER_RE.fullmatch(
                text
            ):
                continue

            if is_toc_line(text):
                continue

            if (
                lower in repeated_short_text
                and len(text) < 120
            ):
                continue

            if is_end_heading(text):
                return sections

            if is_caption(text):
                continue

            if lower.startswith(
                "keywords"
            ):
                continue

            heading = is_heading(
                block
            )

            if heading:
                if not started:
                    if (
                        NUMBERED_HEADING_RE.match(
                            text
                        )
                        or
                        lower in {
                            "abstract",
                            "introduction",
                            "background",
                            "method",
                            "methods",
                            "methodology",
                            "materials and methods",
                            "text",
                        }
                        or
                        (
                            block["bold"]
                            and
                            block["font"] >= 7.5
                            and
                            block["page"] >= 2
                        )
                    ):
                        started = True
                    else:
                        continue

                current = {
                    "heading": text,
                    "page": block["page"],
                    "paragraphs": [],
                }

                sections.append(
                    current
                )

                continue

            if not started:
                continue

            if is_standalone_equation(
                text
            ):
                continue

            if (
                count_words(text) < 5
                and block["font"] < 8.5
            ):
                continue

            current["paragraphs"].append({
                "text": text,
                "page": block["page"],
                "font": block["font"],
                "bold": block["bold"],
            })

    return sections

## 3. Create complete-paragraph chunks within each subsection

In [10]:
def create_paper_chunks(
    sections,
    min_words=MIN_WORDS,
):
    """
    Chunk within each subsection only.

    Rules:
      1. A chunk contains complete paragraphs from one subsection.
      2. Accumulate paragraphs until >= min_words.
      3. A subsection with < min_words total becomes one chunk.
      4. A final undersized remainder is merged into the preceding
         chunk of the same subsection.
      5. No chunk crosses a subsection boundary.
      6. No paragraph is split.
    """
    chunks = []

    for section_index, section in enumerate(
        sections,
        start=1,
    ):
        paragraphs = [
            p
            for p in section["paragraphs"]
            if count_words(
                p["text"]
            ) > 0
        ]

        if not paragraphs:
            continue

        subsection_word_count = sum(
            count_words(
                p["text"]
            )
            for p in paragraphs
        )

        if subsection_word_count < min_words:
            chunks.append({
                "section_index": section_index,
                "section_heading": section["heading"],
                "paragraph_start": 1,
                "paragraph_end": len(
                    paragraphs
                ),
                "paragraph_count": len(
                    paragraphs
                ),
                "word_count": subsection_word_count,
                "start_page": paragraphs[0]["page"],
                "end_page": paragraphs[-1]["page"],
                "text": "\n\n".join(
                    p["text"]
                    for p in paragraphs
                ),
            })
            continue

        current = []
        current_words = 0
        subsection_chunks = []

        for paragraph_index, paragraph in enumerate(
            paragraphs,
            start=1,
        ):
            current.append(
                paragraph
            )

            current_words += count_words(
                paragraph["text"]
            )

            if current_words >= min_words:
                subsection_chunks.append({
                    "section_index": section_index,
                    "section_heading": section["heading"],
                    "paragraph_start": (
                        paragraph_index
                        - len(current)
                        + 1
                    ),
                    "paragraph_end": paragraph_index,
                    "paragraph_count": len(current),
                    "word_count": current_words,
                    "start_page": current[0]["page"],
                    "end_page": current[-1]["page"],
                    "text": "\n\n".join(
                        p["text"]
                        for p in current
                    ),
                })

                current = []
                current_words = 0

        if current:
            remainder_text = "\n\n".join(
                p["text"]
                for p in current
            )

            if subsection_chunks:
                previous = (
                    subsection_chunks[-1]
                )

                previous[
                    "paragraph_end"
                ] = len(
                    paragraphs
                )

                previous[
                    "paragraph_count"
                ] += len(current)

                previous[
                    "word_count"
                ] += current_words

                previous[
                    "end_page"
                ] = current[-1]["page"]

                previous["text"] += (
                    "\n\n"
                    + remainder_text
                )

            else:
                subsection_chunks.append({
                    "section_index": section_index,
                    "section_heading": section["heading"],
                    "paragraph_start": 1,
                    "paragraph_end": len(
                        paragraphs
                    ),
                    "paragraph_count": len(
                        paragraphs
                    ),
                    "word_count": current_words,
                    "start_page": current[0]["page"],
                    "end_page": current[-1]["page"],
                    "text": remainder_text,
                })

        chunks.extend(
            subsection_chunks
        )

    for chunk_id, chunk in enumerate(
        chunks,
        start=1,
    ):
        chunk["chunk_id"] = chunk_id

    return chunks

## 4. Teacher-forced LM scoring helper

In [11]:
import torch
import torch.nn.functional as F

def score_text_lm(text):
    text_for_model = text.lower() if LOWERCASE_FOR_LM else text
    char_count = len(text_for_model)
    input_ids = tokenizer(
        text_for_model,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(DEVICE)
    sequence_length = input_ids.size(1)

    if sequence_length < 2 or char_count == 0:
        return {
            "total_bits": np.nan,
            "scored_tokens": 0,
            "bits_per_token": np.nan,
            "bpc": np.nan,
            "model_tokens": sequence_length,
            "character_count": char_count,
        }

    stride = min(STRIDE, max(1, MODEL_CONTEXT - 1))
    total_nll_nats = 0.0
    scored_tokens = 0
    previous_end = 0

    with torch.inference_mode():
        for end in range(min(stride, sequence_length), sequence_length + stride, stride):
            end = min(end, sequence_length)
            begin = max(0, end - MODEL_CONTEXT)
            input_slice = input_ids[:, begin:end]
            target_ids = input_slice.clone()
            newly_scored = end - previous_end
            if newly_scored < input_slice.size(1):
                target_ids[:, :-newly_scored] = -100

            outputs = model(input_ids=input_slice)
            logits = outputs.logits[:, :-1, :].contiguous()
            labels = target_ids[:, 1:].contiguous()
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
                reduction="sum",
            )
            total_nll_nats += float(loss_sum.item())
            scored_tokens += int((labels != -100).sum().item())
            previous_end = end
            if end >= sequence_length:
                break

    total_bits = total_nll_nats / math.log(2)
    return {
        "total_bits": total_bits,
        "scored_tokens": scored_tokens,
        "bits_per_token": total_bits / scored_tokens if scored_tokens else np.nan,
        "bpc": total_bits / char_count,
        "model_tokens": sequence_length,
        "character_count": char_count,
    }

## 5. Fixed compression reference and compressor definitions

In [12]:
import bz2
import gc
import hashlib
import lzma
import math
import os
import re
import time
import zlib
from pathlib import Path

import requests
import zstandard as zstd
from scipy.stats import spearmanr, pearsonr, rankdata

REFERENCE_URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"
REFERENCE_NAME = "The Adventures of Sherlock Holmes (Project Gutenberg eBook 1661)"
REFERENCE_CHAR_LIMIT = 80000
REFERENCE_CACHE_PATH = OUTPUT_ROOT / "neutral_english_reference.txt"

def load_neutral_reference():
    if REFERENCE_CACHE_PATH.exists():
        raw = REFERENCE_CACHE_PATH.read_text(encoding="utf-8", errors="replace")
    else:
        response = requests.get(REFERENCE_URL, timeout=90)
        response.raise_for_status()
        raw = response.text
        start = re.search(
            r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*",
            raw, flags=re.IGNORECASE
        )
        if start:
            raw = raw[start.end():]
        end = re.search(
            r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*",
            raw, flags=re.IGNORECASE
        )
        if end:
            raw = raw[:end.start()]
        raw = raw.strip()
        REFERENCE_CACHE_PATH.write_text(raw[:REFERENCE_CHAR_LIMIT], encoding="utf-8")
        raw = raw[:REFERENCE_CHAR_LIMIT]
    if len(raw) < 50000:
        raise RuntimeError(
            f"Reference text has only {len(raw)} characters. Check download/cache: {REFERENCE_CACHE_PATH}"
        )
    return raw[:REFERENCE_CHAR_LIMIT]

REFERENCE_TEXT = load_neutral_reference()
REFERENCE_BYTES = REFERENCE_TEXT.encode("utf-8")
print(f"Fixed compression reference: {REFERENCE_NAME}")
print(f"Reference: {REFERENCE_URL}")
print(f"Reference length: {len(REFERENCE_TEXT):,} characters; selected before examining results.")

def c_deflate(data: bytes) -> int:
    comp = zlib.compressobj(level=9, wbits=-15)
    return len(comp.compress(data) + comp.flush())

def c_lzma(data: bytes) -> int:
    filters = [{"id": lzma.FILTER_LZMA2, "preset": 9 | lzma.PRESET_EXTREME}]
    return len(lzma.compress(data, format=lzma.FORMAT_RAW, filters=filters))

def c_bz2(data: bytes) -> int:
    return len(bz2.compress(data, compresslevel=9))

ZSTD_COMPRESSOR = zstd.ZstdCompressor(level=19)

def c_zstd(data: bytes) -> int:
    return len(ZSTD_COMPRESSOR.compress(data))

COMPRESSORS = {
    "plain_deflate": c_deflate,
    "plain_lzma": c_lzma,
    "plain_bz2": c_bz2,
    "plain_zstd": c_zstd,
}

PRIMED_COMPRESSORS = {
    "primed_deflate": c_deflate,
    "primed_lzma": c_lzma,
}
REFERENCE_COMPRESSED_SIZES = {
    method: comp(REFERENCE_BYTES)
    for method, comp in PRIMED_COMPRESSORS.items()
}

def compression_bpc(text: str, method: str) -> float:
    n_chars = len(text)
    if n_chars == 0:
        return np.nan
    data = text.encode("utf-8")
    if method in COMPRESSORS:
        compressed_size = COMPRESSORS[method](data)
    elif method in PRIMED_COMPRESSORS:
        compressed_size = PRIMED_COMPRESSORS[method](REFERENCE_BYTES + data)
        compressed_size -= REFERENCE_COMPRESSED_SIZES[method]
    else:
        raise KeyError(f"Unknown compressor method: {method}")
    # Character denominator matches the study's BPC definition; do not use byte length.
    return 8.0 * compressed_size / n_chars

COMPRESSION_METHODS = [
    "plain_deflate", "plain_lzma", "plain_bz2", "plain_zstd",
    "primed_deflate", "primed_lzma",
]
METHOD_KIND = {
    **{key: "language_model" for key in MODEL_NAMES},
    **{key: "compressor" for key in COMPRESSION_METHODS},
}
MEASURES = ["mean_bpc", "variance_bpc", "skewness_bpc", "excess_kurtosis_bpc"]

Fixed compression reference: The Adventures of Sherlock Holmes (Project Gutenberg eBook 1661)
Reference: https://www.gutenberg.org/cache/epub/1661/pg1661.txt
Reference length: 80,000 characters; selected before examining results.


## 6. Build the analysis chunk manifest

In [13]:
CORPUS_PREFIX = "paper"

chunk_records = []
chunking_status_rows = []

for paper in paper_manifest.itertuples(index=False):
    paper_index = int(paper.paper_index)
    pdf_path = Path(paper.path)
    paper_id = paper.paper_id
    paper_title = str(paper.title) if pd.notna(paper.title) and str(paper.title).strip() else pdf_path.stem
    paper_author = str(paper.author) if pd.notna(paper.author) else ""
    print(f"[{paper_index:02d}/{len(paper_manifest):02d}] {pdf_path.name}")

    try:
        sections = extract_paper_sections(pdf_path)
        if not sections:
            raise RuntimeError("No paper sections could be reconstructed.")
        chunks = create_paper_chunks(sections, MIN_WORDS)
        if not chunks:
            raise RuntimeError("No chunks were created.")

        for chunk in chunks:
            text = chunk.get("text", "")
            chunk_records.append({
                "source_id": paper_id,
                "source_name": paper_title,
                "chunk_id": chunk["chunk_id"],
                "text": text,
                "word_count": int(chunk.get("word_count", count_words(text))),
                "character_count": len(text),
                "source_pdf": str(pdf_path),
                "paper_index": paper_index,
                "paper_author": paper_author,
                "section_index": chunk.get("section_index", np.nan),
                "section_heading": chunk.get("section_heading", ""),
                "paragraph_start": chunk.get("paragraph_start", np.nan),
                "paragraph_end": chunk.get("paragraph_end", np.nan),
                "start_page": chunk.get("start_page", np.nan),
                "end_page": chunk.get("end_page", np.nan),
            })
        chunking_status_rows.append({
            "source_id": paper_id, "source_name": paper_title,
            "source_pdf": str(pdf_path), "status": "READY",
            "n_sections": len(sections), "n_chunks": len(chunks), "error": ""
        })
        print(f"  sections={len(sections)}, chunks={len(chunks)}")

    except Exception as exc:
        chunking_status_rows.append({
            "source_id": paper_id, "source_name": paper_title,
            "source_pdf": str(pdf_path), "status": "ERROR",
            "n_sections": np.nan, "n_chunks": 0,
            "error": f"{type(exc).__name__}: {exc}"
        })
        print(f"  [ERROR] {type(exc).__name__}: {exc}")

chunk_manifest_df = pd.DataFrame(chunk_records)
chunking_status_df = pd.DataFrame(chunking_status_rows)
chunk_manifest_path = OUTPUT_ROOT / "paper_chunk_manifest.csv"
chunking_status_path = OUTPUT_ROOT / "paper_chunking_status.csv"
chunk_manifest_df.drop(columns=["text"]).to_csv(chunk_manifest_path, index=False, encoding="utf-8-sig")
chunking_status_df.to_csv(chunking_status_path, index=False, encoding="utf-8-sig")

if chunk_manifest_df.empty:
    raise RuntimeError("No chunks were reconstructed; check the dataset and chunking logs.")

print(f"Reconstructed {len(chunk_manifest_df)} chunks from {chunk_manifest_df['source_id'].nunique()} papers.")
print("Chunk manifest:", chunk_manifest_path)

[01/29] 1-s2.0-S0022440525001013-main.pdf
  sections=14, chunks=53
[02/29] 1-s2.0-S0022440525001189-main.pdf
  sections=15, chunks=50
[03/29] 1-s2.0-S0140197121001330-main.pdf
  sections=6, chunks=19
[04/29] 1-s2.0-S0140197121001366-main.pdf
  sections=7, chunks=31
[05/29] 1-s2.0-S0160738326001635-main.pdf
  sections=11, chunks=32
[06/29] 1-s2.0-S0160738326001684-main.pdf
  sections=7, chunks=11
[07/29] 1-s2.0-S0190740926000447-main.pdf
  sections=7, chunks=51
[08/29] 1-s2.0-S019074092600071X-main.pdf
  sections=5, chunks=40
[09/29] 1-s2.0-S0195666326003582-main.pdf
  sections=9, chunks=25
[10/29] 1-s2.0-S0195666326003594-main.pdf
  sections=29, chunks=37
[11/29] 1-s2.0-S026151772600124X-main.pdf
  sections=10, chunks=12
[12/29] 1-s2.0-S0261517726001305-main.pdf
  sections=7, chunks=48
[13/29] 1-s2.0-S0277953626009718-main.pdf
  sections=6, chunks=18
[14/29] 1-s2.0-S0277953626009871-main.pdf
  sections=9, chunks=29
[15/29] 1-s2.0-S036013152600182X-main.pdf
  sections=8, chunks=27
[16/2

## 7. Compute compression-BPC baselines

In [14]:
compression_chunk_rows = []

for row in chunk_records:
    base = {
        "source_id": row["source_id"],
        "source_name": row["source_name"],
        "chunk_id": row["chunk_id"],
        "character_count": len(row["text"]),
        "word_count": row["word_count"],
    }
    for method in COMPRESSION_METHODS:
        try:
            value = compression_bpc(row["text"], method)
        except Exception as exc:
            print(f"[COMPRESSION ERROR] {row['source_name']} chunk {row['chunk_id']} / {method}: {exc}")
            value = np.nan
        compression_chunk_rows.append({
            **base,
            "method": method,
            "bpc": value,
            "compressor_status": "SUCCESS" if np.isfinite(value) else "ERROR_OR_NONFINITE",
        })

compression_chunk_df = pd.DataFrame(compression_chunk_rows)
compression_chunk_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_compression_chunk_scores.csv"
compression_chunk_df.to_csv(compression_chunk_path, index=False, encoding="utf-8-sig")
print("Compression chunk scores saved:", compression_chunk_path)
print("Compression methods:", ", ".join(COMPRESSION_METHODS))

Compression chunk scores saved: /kaggle/working/paper_compression_baseline/paper_compression_chunk_scores.csv
Compression methods: plain_deflate, plain_lzma, plain_bz2, plain_zstd, primed_deflate, primed_lzma


## 8. Score the same chunks with the four language models

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def infer_model_context_length(tokenizer, model):
    candidates = [
        getattr(tokenizer, "model_max_length", None),
        getattr(model.config, "n_positions", None),
        getattr(model.config, "max_position_embeddings", None),
        getattr(model.config, "max_sequence_length", None),
    ]
    valid = [int(x) for x in candidates if isinstance(x, int) and 1 < x < 100000]
    return min(valid) if valid else 1024

lm_chunk_rows = []
model_status_rows = []

for model_key, model_name in MODEL_NAMES.items():
    model_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_lm_chunk_scores_{model_key}.csv"
    print(f"\n=== Loading {model_key}: {model_name} ===")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            low_cpu_mem_usage=True,
        )
        model.to(DEVICE)
        model.eval()
        if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        MODEL_CONTEXT = infer_model_context_length(tokenizer, model)
        print("Context length:", MODEL_CONTEXT, "| Device:", DEVICE)

        this_model_rows = []
        for count, row in enumerate(chunk_records, start=1):
            try:
                score = score_text_lm(row["text"])
                this_model_rows.append({
                    "source_id": row["source_id"],
                    "source_name": row["source_name"],
                    "chunk_id": row["chunk_id"],
                    "method": model_key,
                    "model_name": model_name,
                    "bpc": score["bpc"],
                    "bits_per_token": score["bits_per_token"],
                    "total_bits": score["total_bits"],
                    "scored_tokens": score["scored_tokens"],
                    "character_count": score["character_count"],
                    "word_count": row["word_count"],
                    "context_length": MODEL_CONTEXT,
                })
            except Exception as exc:
                this_model_rows.append({
                    "source_id": row["source_id"],
                    "source_name": row["source_name"],
                    "chunk_id": row["chunk_id"],
                    "method": model_key,
                    "model_name": model_name,
                    "bpc": np.nan,
                    "bits_per_token": np.nan,
                    "total_bits": np.nan,
                    "scored_tokens": 0,
                    "character_count": len(row["text"]),
                    "word_count": row["word_count"],
                    "context_length": MODEL_CONTEXT,
                    "error": f"{type(exc).__name__}: {exc}",
                })
            if count % 100 == 0:
                print(f"  Scored {count}/{len(chunk_records)} chunks")

        model_df = pd.DataFrame(this_model_rows)
        model_df.to_csv(model_path, index=False, encoding="utf-8-sig")
        lm_chunk_rows.extend(this_model_rows)
        model_status_rows.append({
            "method": model_key,
            "model_name": model_name,
            "status": "SUCCESS",
            "n_chunks": len(model_df),
            "n_valid_bpc": int(np.isfinite(pd.to_numeric(model_df["bpc"], errors="coerce")).sum()),
            "context_length": MODEL_CONTEXT,
            "error": "",
        })
        print("Saved:", model_path)

        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        model_status_rows.append({
            "method": model_key,
            "model_name": model_name,
            "status": "FAILED",
            "n_chunks": 0,
            "n_valid_bpc": 0,
            "context_length": np.nan,
            "error": f"{type(exc).__name__}: {exc}",
        })
        print(f"[MODEL FAILED] {model_key}: {type(exc).__name__}: {exc}")
        try:
            del model, tokenizer
        except NameError:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

lm_chunk_df = pd.DataFrame(lm_chunk_rows)
if not lm_chunk_df.empty:
    lm_chunk_df.to_csv(OUTPUT_ROOT / f"{CORPUS_PREFIX}_lm_chunk_scores_all_models.csv", index=False, encoding="utf-8-sig")
else:
    raise RuntimeError("No language-model chunk scores were produced. Check model access, network, and the model_run_status output.")
model_status_df = pd.DataFrame(model_status_rows)
model_status_df.to_csv(OUTPUT_ROOT / f"{CORPUS_PREFIX}_model_run_status.csv", index=False, encoding="utf-8-sig")
display(model_status_df)

successful_lms = [
    k for k in MODEL_NAMES
    if k in set(lm_chunk_df["method"]) and
       lm_chunk_df.loc[lm_chunk_df["method"].eq(k), "bpc"].notna().any()
]
if len(successful_lms) < 2:
    raise RuntimeError("Fewer than two language models produced valid chunk scores; cross-model comparison cannot be computed.")
print("Successful language models:", successful_lms)


=== Loading gpt2: gpt2 ===


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Context length: 2048 | Device: cuda
  Scored 100/849 chunks
  Scored 200/849 chunks
  Scored 300/849 chunks


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2864 > 2048). Running this sequence through the model will result in indexing errors


  Scored 400/849 chunks
  Scored 500/849 chunks
  Scored 600/849 chunks
  Scored 700/849 chunks
  Scored 800/849 chunks
Saved: /kaggle/working/paper_compression_baseline/paper_lm_chunk_scores_tinyllama_1_1b.csv


,method,model_name,status,n_chunks,n_valid_bpc,context_length,error
0,gpt2,gpt2,SUCCESS,849,849,1024,
1,pythia_410m,EleutherAI/pythia-410m,SUCCESS,849,849,2048,
2,qwen2_5_0_5b,Qwen/Qwen2.5-0.5B,SUCCESS,849,849,32768,
3,tinyllama_1_1b,TinyLlama/TinyLlama-1.1B-Chat-v1.0,SUCCESS,849,849,2048,


Successful language models: ['gpt2', 'pythia_410m', 'qwen2_5_0_5b', 'tinyllama_1_1b']


## 9. Calculate raw and binned source-level moments

In [16]:
from scipy.stats import skew, kurtosis

METHOD_NAMES = {
    "gpt2": "GPT-2",
    "pythia_410m": "Pythia-410M",
    "qwen2_5_0_5b": "Qwen2.5-0.5B",
    "tinyllama_1_1b": "TinyLlama-1.1B",
    "plain_deflate": "Raw Deflate (level 9)",
    "plain_lzma": "Raw LZMA2 (level 9 extreme)",
    "plain_bz2": "BZip2 (level 9)",
    "plain_zstd": "Zstandard (level 19)",
    "primed_deflate": "Primed raw Deflate",
    "primed_lzma": "Primed raw LZMA2",
}

all_score_rows = []
if not lm_chunk_df.empty:
    all_score_rows.extend(lm_chunk_df.to_dict("records"))
all_score_rows.extend(compression_chunk_df.to_dict("records"))
all_chunk_score_df = pd.DataFrame(all_score_rows)
all_chunk_score_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_all_chunk_scores_lm_and_compressors.csv"
all_chunk_score_df.to_csv(all_chunk_score_path, index=False, encoding="utf-8-sig")

source_meta = (
    chunk_manifest_df.groupby(["source_id", "source_name"], dropna=False)
    .agg(n_chunks=("chunk_id", "count"),
         mean_chunk_length_chars=("character_count", "mean"))
    .reset_index()
)

def histogram_frequency(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return pd.DataFrame(columns=["bin_left", "bin_right", "bin_center", "frequency"])
    if len(values) == 1 or np.isclose(values.min(), values.max()):
        center = float(values[0])
        edges = np.array([center - 0.5, center + 0.5], dtype=float)
    else:
        fd_edges = np.histogram_bin_edges(values, bins="fd")
        suggested = max(1, len(fd_edges) - 1)
        n_bins = int(np.clip(suggested, MIN_BINS, MAX_BINS))
        edges = np.linspace(float(values.min()), float(values.max()), n_bins + 1)
    counts, edges = np.histogram(values, bins=edges)
    centers = (edges[:-1] + edges[1:]) / 2
    keep = counts > 0
    return pd.DataFrame({
        "bin_left": edges[:-1][keep],
        "bin_right": edges[1:][keep],
        "bin_center": centers[keep],
        "frequency": counts[keep].astype(int),
    })

def moments_from_values(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return {"n_valid_chunks": 0, "n_histogram_bins": 0,
                "mean_bpc": np.nan, "variance_bpc": np.nan,
                "skewness_bpc": np.nan, "excess_kurtosis_bpc": np.nan,
                "third_central_moment_bpc": np.nan, "fourth_central_moment_bpc": np.nan}
    mean = float(np.mean(values))
    centered = values - mean
    m2 = float(np.mean(centered ** 2))
    m3 = float(np.mean(centered ** 3))
    m4 = float(np.mean(centered ** 4))
    skew_value = m3 / (m2 ** 1.5) if m2 > 0 else np.nan
    excess = m4 / (m2 ** 2) - 3.0 if m2 > 0 else np.nan
    return {
        "n_valid_chunks": n, "n_histogram_bins": np.nan,
        "mean_bpc": mean, "variance_bpc": m2,
        "skewness_bpc": float(skew_value) if np.isfinite(skew_value) else np.nan,
        "excess_kurtosis_bpc": float(excess) if np.isfinite(excess) else np.nan,
        "third_central_moment_bpc": m3,
        "fourth_central_moment_bpc": m4,
    }

def moments_from_histogram(hist):
    if hist.empty:
        return moments_from_values([])
    centers = hist["bin_center"].to_numpy(dtype=float)
    weights = hist["frequency"].to_numpy(dtype=float)
    n = int(weights.sum())
    mean = float(np.sum(weights * centers) / n)
    centered = centers - mean
    m2 = float(np.sum(weights * centered**2) / n)
    m3 = float(np.sum(weights * centered**3) / n)
    m4 = float(np.sum(weights * centered**4) / n)
    return {
        "n_valid_chunks": n, "n_histogram_bins": int(len(hist)),
        "mean_bpc": mean, "variance_bpc": m2,
        "skewness_bpc": float(m3 / m2**1.5) if m2 > 0 else np.nan,
        "excess_kurtosis_bpc": float(m4 / m2**2 - 3.0) if m2 > 0 else np.nan,
        "third_central_moment_bpc": m3,
        "fourth_central_moment_bpc": m4,
    }

method_values = {
    method: (
        all_chunk_score_df.loc[all_chunk_score_df["method"].eq(method)]
        .groupby("source_id")["bpc"]
        .apply(lambda s: pd.to_numeric(s, errors="coerce").to_numpy(dtype=float))
        .to_dict()
    )
    for method in all_chunk_score_df["method"].dropna().unique()
}

moment_rows = []
frequency_rows = []
for src in source_meta.itertuples(index=False):
    for method, by_source in method_values.items():
        values = np.asarray(by_source.get(src.source_id, []), dtype=float)
        values = values[np.isfinite(values)]
        raw_stats = moments_from_values(values)
        common = {
            "source_id": src.source_id,
            "source_name": src.source_name,
            "method": method,
            "method_name": METHOD_NAMES.get(method, method),
            "method_type": METHOD_KIND.get(method, "unknown"),
            "n_chunks": int(src.n_chunks),
            "n_valid_chunks": len(values),
            "mean_chunk_length_chars": float(src.mean_chunk_length_chars),
        }
        moment_rows.append({
            **common, "moment_estimator": "raw", **raw_stats,
        })
        hist = histogram_frequency(values)
        for row in hist.to_dict("records"):
            frequency_rows.append({
                "source_id": src.source_id, "source_name": src.source_name,
                "method": method, "method_name": METHOD_NAMES.get(method, method),
                **row,
            })
        binned_stats = moments_from_histogram(hist)
        moment_rows.append({
            **common, "moment_estimator": "binned_fd_capped", **binned_stats,
        })

source_moments_df = pd.DataFrame(moment_rows)
frequency_df = pd.DataFrame(frequency_rows)

raw_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_source_moments_raw.csv"
binned_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_source_moments_binned_fd.csv"
freq_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_frequency_distributions_fd.csv"
source_moments_df[source_moments_df["moment_estimator"].eq("raw")].to_csv(raw_path, index=False, encoding="utf-8-sig")
source_moments_df[source_moments_df["moment_estimator"].eq("binned_fd_capped")].to_csv(binned_path, index=False, encoding="utf-8-sig")
frequency_df.to_csv(freq_path, index=False, encoding="utf-8-sig")
print("Raw source moments:", raw_path)
print("Binned source moments:", binned_path)
print("Frequency distributions:", freq_path)
display(source_moments_df.head())

Raw source moments: /kaggle/working/paper_compression_baseline/paper_source_moments_raw.csv
Binned source moments: /kaggle/working/paper_compression_baseline/paper_source_moments_binned_fd.csv
Frequency distributions: /kaggle/working/paper_compression_baseline/paper_frequency_distributions_fd.csv


,source_id,source_name,method,method_name,method_type,n_chunks,n_valid_chunks,mean_chunk_length_chars,moment_estimator,n_histogram_bins,mean_bpc,variance_bpc,skewness_bpc,excess_kurtosis_bpc,third_central_moment_bpc,fourth_central_moment_bpc
0,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,gpt2,GPT-2,language_model,53,53,1964.811321,raw,NaN,1.043742,0.010627,0.638752,0.214442,0.000700,0.000363
1,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,gpt2,GPT-2,language_model,53,53,1964.811321,binned_fd_capped,7.0,1.039717,0.009564,0.588549,-0.045610,0.000551,0.000270
2,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,pythia_410m,Pythia-410M,language_model,53,53,1964.811321,raw,NaN,0.906976,0.006759,0.761951,0.930393,0.000423,0.000180
3,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,pythia_410m,Pythia-410M,language_model,53,53,1964.811321,binned_fd_capped,11.0,0.908133,0.006364,0.757098,0.659574,0.000384,0.000148
4,paper_01_1_s2_0_S0022440525001013_main,Reporting guidelines for single-case experimen...,qwen2_5_0_5b,Qwen2.5-0.5B,language_model,53,53,1964.811321,raw,NaN,0.873627,0.006887,0.926622,1.190517,0.000530,0.000199


## 10. Compare the compression baselines with language models

In [17]:
from itertools import combinations

def safe_spearman(x, y, min_n=3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    value = spearmanr(x, y).statistic
    return float(value) if np.isfinite(value) else np.nan

def mean_finite(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else np.nan

successful_lms = [m for m in MODEL_NAMES if m in set(source_moments_df["method"])]
if len(successful_lms) < 2:
    raise RuntimeError("At least two successful language models are required.")

correlation_rows = []
summary_rows = []
partial_rows = []
length_rows = []
coupling_rows = []

for estimator in ["raw", "binned_fd_capped"]:
    est_df = source_moments_df[source_moments_df["moment_estimator"].eq(estimator)]
    for measure in MEASURES:
        wide = est_df.pivot_table(index="source_id", columns="method", values=measure, aggfunc="first")
        # Pairwise correlations as reported in the paper: all LM-LM pairs.
        lm_pair_list = list(combinations(successful_lms, 2))
        for a, b in lm_pair_list:
            pair = wide.reindex(columns=[a, b]).dropna()
            rho = safe_spearman(pair[a], pair[b]) if not pair.empty else np.nan
            correlation_rows.append({
                "moment_estimator": estimator, "measure": measure,
                "comparison_type": "LM-LM", "method_1": a, "method_2": b,
                "n_sources": len(pair), "spearman_rho": rho,
            })

        # Compressor-to-LM correlations.
        for comp_method in COMPRESSION_METHODS:
            if comp_method not in wide.columns:
                continue
            for lm_method in successful_lms:
                pair = wide.reindex(columns=[comp_method, lm_method]).dropna()
                rho = safe_spearman(pair[comp_method], pair[lm_method]) if not pair.empty else np.nan
                correlation_rows.append({
                    "moment_estimator": estimator, "measure": measure,
                    "comparison_type": "compressor-LM", "method_1": comp_method,
                    "method_2": lm_method, "n_sources": len(pair), "spearman_rho": rho,
                })

        # Paired bootstrap on delta = mean(LM-LM rho) - mean(compressor-LM rho).
        for comp_method in COMPRESSION_METHODS:
            required = successful_lms + [comp_method]
            if not all(m in wide.columns for m in required):
                continue
            matched = wide[required].dropna()
            n = len(matched)
            point_lm_rhos = [
                safe_spearman(matched[a].to_numpy(), matched[b].to_numpy())
                for a, b in lm_pair_list
            ]
            point_comp_rhos = [
                safe_spearman(matched[comp_method].to_numpy(), matched[lm].to_numpy())
                for lm in successful_lms
            ]
            lm_mean_rho = mean_finite(point_lm_rhos)
            comp_mean_rho = mean_finite(point_comp_rhos)
            delta = lm_mean_rho - comp_mean_rho if np.isfinite(lm_mean_rho) and np.isfinite(comp_mean_rho) else np.nan

            boot_deltas = []
            if n >= 4:
                matrix = matched.to_numpy(dtype=float)
                col_idx = {col: j for j, col in enumerate(required)}
                boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
                for _ in range(BOOTSTRAP_REPLICATES):
                    idx = boot_rng.integers(0, n, size=n)
                    sample = matrix[idx, :]
                    lm_rs = [
                        safe_spearman(sample[:, col_idx[a]], sample[:, col_idx[b]])
                        for a, b in lm_pair_list
                    ]
                    comp_rs = [
                        safe_spearman(sample[:, col_idx[comp_method]], sample[:, col_idx[lm]])
                        for lm in successful_lms
                    ]
                    av_lm = mean_finite(lm_rs)
                    av_comp = mean_finite(comp_rs)
                    if np.isfinite(av_lm) and np.isfinite(av_comp):
                        boot_deltas.append(av_lm - av_comp)

            lower = float(np.percentile(boot_deltas, 2.5)) if len(boot_deltas) >= 1000 else np.nan
            upper = float(np.percentile(boot_deltas, 97.5)) if len(boot_deltas) >= 1000 else np.nan
            summary_rows.append({
                "moment_estimator": estimator, "measure": measure,
                "compressor_method": comp_method,
                "n_sources": n,
                "lm_lm_mean_rho": lm_mean_rho,
                "compressor_lm_mean_rho": comp_mean_rho,
                "delta_lm_lm_minus_compressor_lm": delta,
                "delta_ci_95_lower": lower,
                "delta_ci_95_upper": upper,
                "valid_bootstrap_replicates": len(boot_deltas),
                "mean_rho_threshold": PRE_REGISTERED_MEAN_RHO_THRESHOLD if measure == "mean_bpc" else np.nan,
                "mean_threshold_met": bool(comp_mean_rho >= PRE_REGISTERED_MEAN_RHO_THRESHOLD) if measure == "mean_bpc" and np.isfinite(comp_mean_rho) else np.nan,
                "delta_ci_excludes_zero": bool(lower > 0 or upper < 0) if np.isfinite(lower) and np.isfinite(upper) else np.nan,
            })

        # Source-level coupling of mean and variance per method (length/mean confound checks).
        for method in successful_lms + COMPRESSION_METHODS:
            if method not in wide.columns:
                continue
            pair = est_df[est_df["method"].eq(method)].drop_duplicates("source_id").set_index("source_id")
            if "mean_bpc" in pair and "variance_bpc" in pair:
                valid = pair[["mean_bpc", "variance_bpc", "mean_chunk_length_chars", "n_valid_chunks"]].apply(pd.to_numeric, errors="coerce").dropna()
                if not valid.empty:
                    coupling_rows.append({
                        "moment_estimator": estimator, "method": method,
                        "n_sources": len(valid),
                        "pearson_mean_variance": float(pearsonr(valid["mean_bpc"], valid["variance_bpc"]).statistic) if valid["mean_bpc"].nunique() > 1 and valid["variance_bpc"].nunique() > 1 else np.nan,
                        "spearman_mean_variance": safe_spearman(valid["mean_bpc"], valid["variance_bpc"]),
                    })
                    length_rows.append({
                        "moment_estimator": estimator, "method": method,
                        "n_sources": len(valid),
                        "spearman_variance_vs_mean_chunk_length": safe_spearman(valid["variance_bpc"], valid["mean_chunk_length_chars"]),
                        "spearman_variance_vs_n_chunks": safe_spearman(valid["variance_bpc"], valid["n_valid_chunks"]),
                    })

    # Partial Spearman for variance controlling for each method's own mean rank.
    measure = "variance_bpc"
    wide_var = est_df.pivot_table(index="source_id", columns="method", values=measure, aggfunc="first")
    wide_mean = est_df.pivot_table(index="source_id", columns="method", values="mean_bpc", aggfunc="first")

    def partial_spearman_two_controls(x, y, control_x, control_y):
        arr = np.column_stack([x, y, control_x, control_y]).astype(float)
        arr = arr[np.all(np.isfinite(arr), axis=1)]
        if len(arr) < 5 or any(np.unique(arr[:, i]).size < 2 for i in range(4)):
            return np.nan
        rx, ry, rcx, rcy = [rankdata(arr[:, i]) for i in range(4)]
        design = np.column_stack([np.ones(len(arr)), rcx, rcy])
        rx_resid = rx - design @ np.linalg.lstsq(design, rx, rcond=None)[0]
        ry_resid = ry - design @ np.linalg.lstsq(design, ry, rcond=None)[0]
        if np.std(rx_resid) == 0 or np.std(ry_resid) == 0:
            return np.nan
        return float(np.corrcoef(rx_resid, ry_resid)[0, 1])

    all_methods = successful_lms + COMPRESSION_METHODS
    for a, b in combinations(all_methods, 2):
        if a not in wide_var or b not in wide_var or a not in wide_mean or b not in wide_mean:
            continue
        cols = pd.concat([
            wide_var[[a, b]].rename(columns={a:"var_a", b:"var_b"}),
            wide_mean[[a, b]].rename(columns={a:"mean_a", b:"mean_b"}),
        ], axis=1).dropna()
        if len(cols) < 5:
            continue
        if (a in successful_lms and b in successful_lms) or (a in COMPRESSION_METHODS and b in COMPRESSION_METHODS):
            continue
        rho_partial = partial_spearman_two_controls(
            cols["var_a"].to_numpy(), cols["var_b"].to_numpy(),
            cols["mean_a"].to_numpy(), cols["mean_b"].to_numpy(),
        )
        partial_rows.append({
            "moment_estimator": estimator,
            "method_1": a, "method_2": b,
            "n_sources": len(cols),
            "partial_spearman_variance_controlling_means": rho_partial,
        })

corr_df = pd.DataFrame(correlation_rows)
summary_df = pd.DataFrame(summary_rows)
partial_df = pd.DataFrame(partial_rows)
length_df = pd.DataFrame(length_rows).drop_duplicates()
coupling_df = pd.DataFrame(coupling_rows).drop_duplicates()

corr_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_pairwise_lm_and_compressor_correlations.csv"
summary_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_baseline_vs_lm_summary.csv"
partial_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_variance_partial_spearman.csv"
length_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_variance_length_confound_checks.csv"
coupling_path = OUTPUT_ROOT / f"{CORPUS_PREFIX}_mean_variance_coupling.csv"

corr_df.to_csv(corr_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
partial_df.to_csv(partial_path, index=False, encoding="utf-8-sig")
length_df.to_csv(length_path, index=False, encoding="utf-8-sig")
coupling_df.to_csv(coupling_path, index=False, encoding="utf-8-sig")

print("Saved pairwise correlations:", corr_path)
print("Saved LM-LM vs compressor-LM summaries:", summary_path)
print("Saved partial Spearman variance checks:", partial_path)
print("Saved variance/length confound checks:", length_path)
print("Saved mean-variance coupling:", coupling_path)
display(summary_df)

Saved pairwise correlations: /kaggle/working/paper_compression_baseline/paper_pairwise_lm_and_compressor_correlations.csv
Saved LM-LM vs compressor-LM summaries: /kaggle/working/paper_compression_baseline/paper_baseline_vs_lm_summary.csv
Saved partial Spearman variance checks: /kaggle/working/paper_compression_baseline/paper_variance_partial_spearman.csv
Saved variance/length confound checks: /kaggle/working/paper_compression_baseline/paper_variance_length_confound_checks.csv
Saved mean-variance coupling: /kaggle/working/paper_compression_baseline/paper_mean_variance_coupling.csv


,moment_estimator,measure,compressor_method,n_sources,lm_lm_mean_rho,compressor_lm_mean_rho,delta_lm_lm_minus_compressor_lm,delta_ci_95_lower,delta_ci_95_upper,valid_bootstrap_replicates,mean_rho_threshold,mean_threshold_met,delta_ci_excludes_zero
0,raw,mean_bpc,plain_deflate,29,0.975369,0.453448,0.521921,0.199828,0.911905,5000,0.85,False,True
1,raw,mean_bpc,plain_lzma,29,0.975369,0.376847,0.598522,0.206664,1.018793,5000,0.85,False,True
2,raw,mean_bpc,plain_bz2,29,0.975369,0.339901,0.635468,0.268321,1.048434,5000,0.85,False,True
3,raw,mean_bpc,plain_zstd,29,0.975369,0.374877,0.600493,0.248626,1.001132,5000,0.85,False,True
4,raw,mean_bpc,primed_deflate,29,0.975369,0.689039,0.286330,0.134945,0.518694,5000,0.85,False,True
5,raw,mean_bpc,primed_lzma,29,0.975369,0.767365,0.208005,0.087670,0.416989,5000,0.85,False,True
6,raw,variance_bpc,plain_deflate,29,0.947291,0.535591,0.411700,0.157637,0.734583,5000,NaN,NaN,True
7,raw,variance_bpc,plain_lzma,29,0.947291,0.454680,0.492611,0.200596,0.853784,5000,NaN,NaN,True
8,raw,variance_bpc,plain_bz2,29,0.947291,0.384606,0.562685,0.257140,0.936614,5000,NaN,NaN,True
9,raw,variance_bpc,plain_zstd,29,0.947291,0.550985,0.396305,0.139548,0.722721,5000,NaN,NaN,True


## Outputs and interpretation

The notebook computes both **raw moments from observed chunk scores** and **binned moments from capped Freedman–Diaconis frequency histograms**. The raw moments are the direct empirical moments of the chunk-level BPC values; the binned moments are a sensitivity analysis for the paper's histogram-based implementation.

Primary compression methods are raw Deflate and raw LZMA2, plus their reference-primed variants. BZip2 and Zstandard are reported as unprimed robustness checks. Compression BPC divides by `len(chunk)` in characters, not UTF-8 byte count. Primed scores are conditional size differences and are not clipped if a compressor returns a nonpositive difference.

The comparison summary reports, for each measure and estimator, the mean of the six available LM–LM Spearman correlations, the mean compressor–LM correlation, and their difference with a 95% source-level paired-bootstrap interval. The pre-specified descriptive threshold for a nontrivial mean-BPC baseline is 0.85 for the average compressor–LM correlation. This is an operational threshold, not a universal statistical standard.

Additional CSVs report pairwise correlations, partial Spearman correlations for variance while controlling for both methods' mean rankings, mean–variance coupling, and correlations of variance with mean chunk length and chunk count. These are diagnostics, not proof that compression and language-model scores measure the same underlying construct.